In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.ToTensor()
train_data = MNIST(root='./data', train=True, download=True, transform=transform)   
test_data = MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

class SimpleCNN(nn.Module): 
    def __init__(self):
        super().__init__() 
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), 
            nn.Conv2d(8, 16, kernel_size=3, padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(16 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


# ==========================================
# COMPLETION: Training and Evaluation Loops
# ==========================================

# 1. Initialize model, loss function, and optimizer
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Training Loop
epochs = 5
print("Starting training...")

for epoch in range(epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    
    for images, labels in train_loader:
        # Move data to the active device (CPU or GPU)
        images = images.to(device)
        labels = labels.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

# 3. Evaluation Loop
print("Evaluating on test data...")
model.eval() # Set model to evaluation mode
correct = 0
total = 0

# Disable gradient calculation for testing to save memory and compute
with torch.no_grad(): 
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        
        # Get the index of the max log-probability
        predicted = outputs.argmax(dim=1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy on the 10,000 test images: {accuracy:.2f}%")

Using device: cpu
Starting training...
Epoch [1/5], Loss: 0.3425
Epoch [2/5], Loss: 0.0872
Epoch [3/5], Loss: 0.0633
Epoch [4/5], Loss: 0.0509
Epoch [5/5], Loss: 0.0413
Evaluating on test data...
Test Accuracy on the 10,000 test images: 98.20%
